# Objectives:

The main objective of this notebook is to explore the raw data structurally and obtain the most basic information needed for data cleaning.
Any changes in the dataset in this notebook are purely for exploration and will be neglected in the next notebooks.\
Additional objectives are:
- Are the data types of each column correct?
- Are there missing or "unknown" values? If there are, how much of the data do they represent?
- Are there incorrect values such as negative values in features that should realistically contain only positive values?
- Are there special cases in the data that could contain hidden meanings?

# Code:

## Initializing:

In [1]:
import os
os.chdir('..')

In [2]:
import numpy as np
import pandas as pd

from src.calculators import percent

In [3]:
raw_df = pd.read_csv('data/raw/Online_Retail.csv', encoding='ISO-8859-1')
# Importing the data from Kaggle directly is also an option

In [4]:
# This copy is made just in case we want to reset df without loading the dataset again
df = raw_df.copy()

## Basic exploration:

In [5]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/11 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/11 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/11 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/11 12:50,4.15,12680.0,France


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 66.2 MB


The column InvoiceDate contains data of string type and not datetime64.

In [7]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [8]:
df.duplicated().sum()

np.int64(5268)

Findings & Interpretations:\
Dataset & Data types:\
The dataset contains 541909 transactions (rows) and 8 starting features (columns).\
The dataset contains 5268 duplicated rows.\
The InvoiceNo column has data type string. This is because some invoice numbers start with the letter "C" to identify cancellations.\
The StockCode column has data type string. This could mean that stock codes contain non-numeric characters.\
The InvoiceDate column has data type string instead of datetime64.\
The time in the InvoiceDate column is a 24-hour cycle.\
The CustomerID column has data type float. This is weird since ID data are usually integer or string, but it could be because of the missing values.

Missing Data:\
The only columns to contain missing values are Description and CustomerID.\
Even though CustomerID has a lot of missing values, Country column has no missing values, which probably means that the source of the Country column is the order address and not the customer's profile.

Numbers & Percentages:\
The minimum and maximum values of the Quantity column happen to be the same, only differing in sign.\
The Quantity column's minimum value is negative. If multiple values in the Quantity column are negative too, that might mean returned items or cancellations.\
The UnitPrice column's minimum value is negative. If this occurs a lot then it could mean returned items or refund adjustments.\
The Quantity and UnitPrice columns' 75th percentile is much smaller than the maximum value. This suggests significant right-skewness, and extreme outliers in both columns.

Actions & Plans:\
Investigate the duplicated rows visually to see if they are truly duplicate and decide how to deal with them.\
Investigate further the InvoiceNo column to see if it contains only numeric characters (except for the "C" at the start of cancelled orders' invoices)\
The StockCode column must be investigated to see if product codes actually contain non-numeric characters, what are their patterns, and how to deal with them.\
Turn InvoiceDate to datetime64 to allow flawless extraction of features and help in exploration.\
Investigate the non-missing data in CustomerID column to make sure it doesn't contain decimal values.

Missing descriptions and customer IDs should be investigated and see if there are patterns behind them.\
Investigate if missing descriptions also mean incorrect or ambigiuous stock code.\
Ensure that the Country column contains only countries and with consistent labelling.

Investigate the meaning behind the minimum and maximum values of the Quantity having the same number but with opposite sign.\
Investigate further the negative values in the Quantity and UnitPrice columns and whether or not they are related to cancellations.\
Investigate the big gap in the Quantity and UnitPrice columns. (during EDA, not in this notebook)

## Duplicates:

Let's see what is the percentage of duplicated data:

In [9]:
duplicate_df = df[df.duplicated()]
print('Total duplicated rows:', len(duplicate_df))
print('Percentage of duplicated rows:', percent(len(duplicate_df), len(df)))

Total duplicated rows: 5268
Percentage of duplicated rows: 0.97%


How many invoices contain duplicated transaction rows?

In [10]:
# Creating a variable with the number of unique values of the "InvoiceNo" column that is in the duplicated rows of the dataframe
duplicate_invoices = duplicate_df['InvoiceNo'].nunique()
print('Invoices with duplicates:', duplicate_invoices)
print('Percentage of invoices with duplicates:', percent(duplicate_invoices, df['InvoiceNo'].nunique()))

Invoices with duplicates: 1933
Percentage of invoices with duplicates: 7.46%


~7% of invoices (1933 invoices) contain at least one duplicate transaction row.

---

## Understanding the data:

### Invoice Timestamps:

The InvoiceDate column is stored as a string data type.
Even though the transformation to datetime64 type is typically done in the data cleaning process, it should be done temporarily in this notebook for exploratory feature engineering and better structural understanding of the dataset.

In [11]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceDate'].sample(10)

C:\Users\micro\AppData\Local\Temp\ipykernel_26360\659678354.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


222929   2011-06-12 10:46:00
392165   2011-10-11 16:29:00
55952    2011-01-13 11:26:00
402097   2011-10-17 14:29:00
464380   2011-11-14 09:00:00
66052    2011-01-21 11:18:00
20449    2010-12-09 13:34:00
68695    2011-01-24 09:41:00
317156   2011-08-30 10:39:00
136495   2011-03-28 15:45:00
Name: InvoiceDate, dtype: datetime64[us]

Let's create some temporary exploratory features from the timestamps to help explore the data.

In [12]:
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['Hour'] = df['InvoiceDate'].dt.hour

In [13]:
df['Year'].value_counts()

Year
2011    499428
2010     42481
Name: count, dtype: int64

In [14]:
print('Percentage of transactions in 2011:', percent(len(df['Year'][df['Year'] == 2011]), len(df)))

Percentage of transactions in 2011: 92.16%


~92% of the transactions are in the year 2011.\
\~8% of the transactions are in the year 2010.\
This big difference makes sense since the transactions start in December 2010 (the end of the year 2010).

---

### Unique Values Inspection:

Let's inspect the unique values in the dataset.

In [15]:
len(df)

541909

In [16]:
df.nunique()

InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
Year               2
Month             12
Day               31
Hour              15
dtype: int64

The length of the dataset is greater than the unique values of InvoiceNo, meaning that:
1. Some transactions may have the same invoice number.
2. Rows don't describe a whole order, but instead an item in an order.
3. Some (most) orders have multiple items.

The length of the dataset is greater than the unique values of InvoiceDate, meaning that some transactions may have the same timestamp.\
Unique values of the InvoiceDate column are less than unique values of InvoiceNo, meaning that multiple orders can happen at the same time.

Even though some of the Description column has missing values, the number of unique descriptions is greater than the number of unique stock codes. This means that some stock codes can point to more than one description, which is likely a data entry error or labeling inconsistency.

There are only 4372 unique customer ids, but we can't fairly compare between this number and the number of unique invoice numbers since the CustomerID column contains missing values.\
Let's get the numbers of unique invoice numbers that don't have a missing customer id.

In [17]:
df[~df['CustomerID'].isna()].nunique()

InvoiceNo      22190
StockCode       3684
Description     3896
Quantity         436
InvoiceDate    20460
UnitPrice        620
CustomerID      4372
Country           37
Year               2
Month             12
Day               31
Hour              15
dtype: int64

The number of unique invoice numbers in rows that contain a customer id is still larger than the number of unique customer ids.

---

### Sorting Key:

Let's see if either of InvoiceNo and InvoiceDate are monotonic increasing:

In [18]:
df['InvoiceDate'].is_monotonic_increasing

True

InvoiceDate is monotonic increasing

Since the column InvoiceNo includes alphabetic characters, we must strip it of them first to check whether it's monotonic increasing.

In [19]:
# Creating a series containing invoice numbers without the alphabetic characters (if there are any)
invoices_stripped = df['InvoiceNo'].apply(lambda x: int(x[1:]) if x[0].isalpha() else int(x))

# Getting the unique values to check if invoices are monotonic increasing
invoices_stripped_unique = pd.Series(invoices_stripped.unique()) 
invoices_stripped_unique.is_monotonic_increasing

False

InvoiceNo is NOT monotonic increasing

In [20]:
invoices_stripped_unique.is_monotonic_decreasing

False

InvoiceNo is not monotonic decreasing either.\
Therefore InvoiceNo column is not sorted in any way. Only InvoiceDate is.\
So the dataset is sorted by date and time.

Let's see if all invoices are done by only one customer each:

In [21]:
# Getting the number of unique customers per invoice (sorted descending)
df.groupby('InvoiceNo')['CustomerID'].nunique().sort_values(ascending=False)

InvoiceNo
C581569    1
536365     1
536366     1
536367     1
536368     1
          ..
581435     0
581431     0
581498     0
581497     0
581492     0
Name: CustomerID, Length: 25900, dtype: int64

All transactions within the same invoice are done by only one customer. Except for the invoices having a missing customer id, of course.

---

### Cancellation Invoices:

It's already known that InvoiceNo may contain the letter "C" at the start of an invoice number to indicate a cancelled order, so let's check if there are other alphabetic prefixes present in invoice numbers.

In [22]:
# Applying a lambda function to the InvoiceNo column that returns the value counts of only the first character of every invoice number that contains non-numeric characters
df['InvoiceNo'].apply(lambda x: x[0] if not x.isdigit() else None).dropna().value_counts()

InvoiceNo
C    9288
A       3
Name: count, dtype: int64

There seems to be 3 rows with an invoice that has an "A" prefix.

Let's see if cancelled invoice numbers are repeated without the "C":

In [23]:
# Creating a Series with only cancelled invoice numbers
cancelled_invoices = df['InvoiceNo'].apply(lambda x: x if x[0] == 'C' else None).dropna()

# Removing the "C" at the start of every cancelled invoice number
stripped_cancelled_invoices = cancelled_invoices.apply(lambda x: x[1:])

# Checking if the stripped_cancelled_invoices Series and InvoiceNo column have any elements in common
(stripped_cancelled_invoices.isin(df['InvoiceNo'])).any()

np.False_

Cancelled invoice numbers don't repeat without the "C".

---

### Country:

Let's take a look on the present countries in the dataset:

In [24]:
df['Country'].value_counts()

Country
United Kingdom          495478
Germany                   9495
France                    8557
EIRE                      8196
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               2002
Portugal                  1519
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Unspecified                446
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     297
USA                        291
Hong Kong                  288
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
RSA                         58


There seems to be 446 "unspecified" countries.

In [25]:
# Creating a dataframe where the Country column is "Unspecified"
df_unspecified_country = df[df['Country'] == 'Unspecified']
print('Percentage of unspecified countries:', percent(len(df_unspecified_country), len(df)))

Percentage of unspecified countries: 0.08%


~0.08% of the transactions have an unspecified country (446 rows out of 536,641).

---

### Stock Codes:

In [26]:
stockcodes = pd.Series(df['StockCode'].unique())

The column StockCode is stored as a string data type so let's see if it contains non-digit characters:

In [27]:
# Creating a series with only non-numeric stock codes
non_numeric_stockcodes = stockcodes.apply(lambda x: x if not x.isdigit() else None).dropna()
print('Non-numeric stock codes:', len(non_numeric_stockcodes))
print('Percentage of non-numeric stock codes:', percent(len(non_numeric_stockcodes), len(stockcodes)))

Non-numeric stock codes: 1124
Percentage of non-numeric stock codes: 27.62%


In [28]:
df_non_numeric_stockcodes = df[df['StockCode'].isin(non_numeric_stockcodes)]
print('Rows with non-numeric stock codes:', len(df_non_numeric_stockcodes))
print('Percentage of transactions with non-numeric stock codes:', percent(len(df_non_numeric_stockcodes), len(df)))

Rows with non-numeric stock codes: 54873
Percentage of transactions with non-numeric stock codes: 10.13%


~27.6% of stock codes are non-numeric (contain alphabetic characters)\
~10% of transactions contain non-numeric stock codes.

Let's check for stock codes containing only alphabetic letters.

In [29]:
# Creating a series with only alphabetic stock codes
alphabetic_stockcodes = stockcodes.apply(lambda x: x if x.isalpha() else None).dropna()
print('Alphabetic stock codes:', len(alphabetic_stockcodes))
print('Percentage of alphabetic stock codes:', percent(len(alphabetic_stockcodes), len(stockcodes)))

Alphabetic stock codes: 12
Percentage of alphabetic stock codes: 0.29%


In [30]:
df_alphabetic_stockcodes = df[df['StockCode'].isin(alphabetic_stockcodes)]
print('Rows with alphabetic stock codes:', len(df_alphabetic_stockcodes))
print('Percentage of transactions with alphabetic stock codes:', percent(len(df_alphabetic_stockcodes), len(df)))

Rows with alphabetic stock codes: 2759
Percentage of transactions with alphabetic stock codes: 0.51%


~0.3% of stock codes are alphabetic.\
~0.5% of transactions contain alphabetic stock codes.

---

### Quantity:

In [31]:
df['Quantity'].describe()

count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

The quantity column seems to deviate a lot.

In [32]:
df['Quantity'].quantile(0.95)

np.float64(29.0)

95% of the quantities are less than or equal to 30, which is nowhere near the maximum value (80,955).\
This means the data contains heavy outliers.

Let's check if there are transactions with a quantity of zero.

In [33]:
(df['Quantity'] == 0).sum()

np.int64(0)

There are no transactions with a quantity of zero

#### Negative Quantities:

Let's inspect how many transactions have negative quantities.

In [34]:
df_negative_quantity = df[df['Quantity'] < 0]
print('Rows with transactions with a negative quantity:', len(df_negative_quantity))
print('Percentage of transactions with a negative quantity:', percent(len(df_negative_quantity), len(df)))

Rows with transactions with a negative quantity: 10624
Percentage of transactions with a negative quantity: 1.96%


~2% of the transactions have a negative quantity (10,587 rows).

In [35]:
df_negative_quantity['Quantity'].describe()

count    10624.000000
mean       -45.607210
std       1092.214216
min     -80995.000000
25%        -10.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64

In [36]:
df_negative_quantity['Quantity'].quantile(0.05)

np.float64(-96.0)

Even 95% of negative quantities are more than or equal to -96, which is no where near the lowest negative value (-80,995).

---

### Non-positive Unit Prices:

In [37]:
df['UnitPrice'].describe()

count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64

Let's see how many negative unit prices are there:

In [38]:
df_neg_price = df[df['UnitPrice'] < 0]
print('Rows with a negative unit price:', len(df_neg_price))
print('Percentage of negative unit prices:', percent(len(df_neg_price), len(df)))

Rows with a negative unit price: 2
Percentage of negative unit prices: 4e-04%


There are only 2 transactions with negative unit price.

Let's check how many records have a unit price of zero.

In [39]:
df_zero_price = df[df['UnitPrice'] == 0]
print('Rows with a unit price of zero:', len(df_zero_price))
print('Percentage of unit prices of zero:', percent(len(df_zero_price), len(df)))

Rows with a unit price of zero: 2515
Percentage of unit prices of zero: 0.46%


Only ~0.5% of the transactions contain a unit price of zero.

## Missing Data:

It is already known that only the CustomerID and Description columns contain missing values. Now, let's see how many values are missing in each column.

In [40]:
missing_counts = df.isna().sum()
missing_counts

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
Year                0
Month               0
Day                 0
Hour                0
dtype: int64

In [41]:
customerid_missing_count = missing_counts.loc['CustomerID']
print('Rows with a missing customer ID:', customerid_missing_count)
print('Percentage of transactions with a missing customer ID:', percent(customerid_missing_count, len(df)))

Rows with a missing customer ID: 135080
Percentage of transactions with a missing customer ID: 24.93%


In [42]:
description_missing_count = missing_counts.loc['Description']
print('Rows with a missing description:', description_missing_count)
print('Percentage of transactions with a missing description:', percent(description_missing_count, len(df)))

Rows with a missing description: 1454
Percentage of transactions with a missing description: 0.27%


~25% of the transactions have a missing customer ID (135,080 rows).\
\~0.27% of the transactions have a missing description (1,454 rows).

Let's see if both Description and CustomerID are missing in the same rows.

In [43]:
# Getting the length of the dataframe where both the description and customer ID are missing
len(df[df['Description'].isna() & df['CustomerID'].isna()])

1454

The number of rows where both the description and customer ID are missing is equal to the number of rows where the description is missing, this means that whenever the description is missing, the customer ID is also missing. This also implies that in all incomplete rows, CustomerID is missing.

---

# Conclusions:

The dataset contains 541,909 rows and 8 starting columns.\
The dataset contains 5268 duplicated rows which represents ~0.1 of the data.\
Each row represents a transaction within an invoice, meaning that multiple transactions can be within the same invoice.\
The dataset is ascendingly sorted by timestamps.

The InvoiceDate column is stored as a string data type instead of datetime64.\
The time in the InvoiceDate column is a 24-hour cycle.\
\~92% of the transactions are in the year 2011. And the rest are in 2010\
Some invoices have the same timestamp.

Some invoice numbers start with the letter "C" to identify cancellations and 3 invoice numbers start with the letter "A".
All transactions within the same invoice are done the same customer.\
Invoice numbers starting with “C” (cancelled) do not have a corresponding invoice with the same number without the prefix.

The only columns to contain missing values are Description and CustomerID.\
Even though CustomerID has lots of missing values, Country column has no missing values and only has 446 unspecified country records, which most likely means that the source of the Country column is the order address and not the customer's profile/entry.\
\~25% of the transactions have a missing customer ID (135,037 rows).\
\~0.27% of the transactions have a missing description (1,454 rows).

The Quantity column likely contains heavy outliers.\
There are no transactions with a quantity of zero.\
\~2% of the quantities are negative (10,587 rows).

Only two rows in the dataset contain negative unit prices.\
\~0.5% of the rows contain a unit price of zero.

There are 37 unique countries present in the data. (38 if "unspecified" is included).\
\~0.08% of the transactions have an unspecified country (446 rows). 

Some stock codes can point to more than one description, which is likely a data entry error or data inconsistency.\
\~27.6% of stock codes are non-numeric and the rest consist solely of digits.\
\~10% of transactions contain non-numeric stock codes\
\~0.3% of stock codes are purely alphabetic and they are present in \~0.5% of total transactions.

---